# US Gun Violence — Interactive State Heat Map
### Race & Sex Breakdown · 1999–2024

**Required files:**
- `Underlying_Cause_of_Death__1999-2020.csv` — CDC WONDER, state × year × race × sex (bridged-race)
- `Underlying_Cause_of_Death__2018-2024__Single_Race.csv` — CDC WONDER, state × year × race × sex (single-race)

**Merge strategy:** 2018-2024 file is authoritative for 2018 onward. 1999–2017 comes from the older file.

**Hover tooltips show on every map:**
- Total gun deaths + crude rate per 100,000
- Deaths and % share broken out by every race group
- Deaths and % share broken out by sex (Male / Female)

**⚠️ Death-type note:** These files contain total firearm deaths only — no homicide/suicide/unintentional split. To add that layer, re-query CDC WONDER grouped by ICD-10 cause list (X72–X74 suicide, X93–X95 homicide, W32–W34 unintentional) and drop the resulting CSV into the same folder. See Cell 9 for the merge hook.

---
**Maps produced:**
1. Animated choropleth — year slider 1999→2024
2. Single-year interactive map (configurable)
3. Race-stratified maps — one per race group
4. Male vs. Female rate maps
5. Trend line chart — top states over time
6. Black vs. White disparity chart

In [2]:
# ── CELL 0: INSTALL & IMPORT ─────────────────────────────────────────────────
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ['plotly', 'pandas']:
    try:
        __import__(pkg)
    except ImportError:
        _pip(pkg)

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

Imports OK


In [3]:
# ── CELL 1: CONFIG ────────────────────────────────────────────────────────────
FILE_99_20 = 'Underlying_Cause_of_Death__1999-2020.csv'
FILE_18_24 = 'Underlying_Cause_of_Death__2018-2024__Single_Race.csv'

# Optional: path to a CDC WONDER cause-type CSV (homicide/suicide/unintentional)
# Leave as None until you have it
FILE_CAUSE = None   # e.g. 'Underlying_Cause_of_Death__by_Cause.csv'

FIPS_TO_ABBR = {
    '01':'AL','02':'AK','04':'AZ','05':'AR','06':'CA','08':'CO','09':'CT','10':'DE',
    '11':'DC','12':'FL','13':'GA','15':'HI','16':'ID','17':'IL','18':'IN','19':'IA',
    '20':'KS','21':'KY','22':'LA','23':'ME','24':'MD','25':'MA','26':'MI','27':'MN',
    '28':'MS','29':'MO','30':'MT','31':'NE','32':'NV','33':'NH','34':'NJ','35':'NM',
    '36':'NY','37':'NC','38':'ND','39':'OH','40':'OK','41':'OR','42':'PA','44':'RI',
    '45':'SC','46':'SD','47':'TN','48':'TX','49':'UT','50':'VT','51':'VA','53':'WA',
    '54':'WV','55':'WI','56':'WY',
}

COLORSCALE = [
    [0.00, '#f7fbff'],
    [0.20, '#c6dbef'],
    [0.40, '#6baed6'],
    [0.65, '#2171b5'],
    [1.00, '#08306b'],
]

# Human-readable race labels keyed on CDC column name
RACE_LABELS = {
    'deaths_Black or African American'               : 'Black / African American',
    'deaths_White'                                   : 'White',
    'deaths_American Indian or Alaska Native'        : 'American Indian / AK Native',
    'deaths_Asian or Pacific Islander'               : 'Asian / Pacific Islander',
    'deaths_Asian'                                   : 'Asian',
    'deaths_Native Hawaiian or Other Pacific Islander': 'Native Hawaiian / Pacific Islander',
}

RACE_COLORS = {
    'deaths_Black or African American'               : 'Reds',
    'deaths_White'                                   : 'Blues',
    'deaths_American Indian or Alaska Native'        : 'Oranges',
    'deaths_Asian or Pacific Islander'               : 'Purples',
    'deaths_Asian'                                   : 'Purples',
    'deaths_Native Hawaiian or Other Pacific Islander': 'Greens',
}

print('Config ready.')

Config ready.


In [4]:
# ── CELL 2: LOAD RAW FILES ────────────────────────────────────────────────────
def _load_wonder(path):
    """Load a CDC WONDER tab-or-comma CSV, stripping the footer notes section."""
    with open(path) as fh:
        lines = fh.readlines()
    footer = next(
        (i for i, l in enumerate(lines) if l.strip().strip('"').startswith('---')),
        None
    )
    df = pd.read_csv(path, sep=None, engine='python',
                     nrows=footer - 1 if footer else None)
    df = df[df['State'].notna()].copy()
    df['Deaths']     = pd.to_numeric(df['Deaths'],     errors='coerce')
    df['Population'] = pd.to_numeric(df['Population'], errors='coerce')
    return df


print(f'Loading {FILE_99_20} ...')
raw_99 = _load_wonder(FILE_99_20)
raw_99['race'] = raw_99['Race']            # bridged-race column

print(f'Loading {FILE_18_24} ...')
raw_18 = _load_wonder(FILE_18_24)
raw_18['race'] = raw_18['Single Race 6']   # single-race column

print(f'  1999-2020 → {len(raw_99):,} rows | {raw_99["Year"].min()}–{raw_99["Year"].max()} | races: {sorted(raw_99["race"].dropna().unique())}')
print(f'  2018-2024 → {len(raw_18):,} rows | {raw_18["Year"].min()}–{raw_18["Year"].max()} | races: {sorted(raw_18["race"].dropna().unique())}')

Loading Underlying_Cause_of_Death__1999-2020.csv ...


FileNotFoundError: [Errno 2] No such file or directory: 'Underlying_Cause_of_Death__1999-2020.csv'

In [ ]:
# ── CELL 3: MERGE INTO ONE LONGITUDINAL DATASET ───────────────────────────────
# 1999-2017  → bridged-race file (4 race groups)
# 2018-2024  → single-race file  (5 race groups, more precise)

KEEP = ['State', 'State Code', 'Year', 'race', 'Sex', 'Deaths', 'Population']

df_long = pd.concat([
    raw_99[raw_99['Year'] <= 2017][KEEP],
    raw_18[KEEP],
], ignore_index=True)

print(f'Combined: {len(df_long):,} rows | {df_long["Year"].min()}–{df_long["Year"].max()}')

In [5]:
# ── CELL 4: BUILD ANALYSIS FRAME (state × year) ───────────────────────────────
# One row per state + year containing total deaths, race breakdown, sex breakdown

# --- Total deaths ---
totals = (
    df_long.groupby(['State', 'State Code', 'Year'])['Deaths']
    .sum().reset_index()
    .rename(columns={'Deaths': 'total_deaths'})
)

# --- Deaths by race ---
race_piv = (
    df_long.groupby(['State', 'State Code', 'Year', 'race'])['Deaths']
    .sum().unstack('race').reset_index()
)
race_piv.columns = [
    c if c in ['State', 'State Code', 'Year'] else f'deaths_{c}'
    for c in race_piv.columns
]

# --- Deaths by sex ---
sex_piv = (
    df_long.groupby(['State', 'State Code', 'Year', 'Sex'])['Deaths']
    .sum().unstack('Sex').reset_index()
)
sex_piv.columns = [
    c if c in ['State', 'State Code', 'Year'] else f'deaths_{c}'
    for c in sex_piv.columns
]

# --- Population (remove race-duplication by dividing by n distinct races) ---
n_races = (
    df_long.groupby(['State', 'State Code', 'Year'])['race']
    .nunique().reset_index(name='n_races')
)
pop = df_long.groupby(['State', 'State Code', 'Year'])['Population'].sum().reset_index()
pop = pop.merge(n_races, on=['State', 'State Code', 'Year'])
pop['Population'] = (pop['Population'] / pop['n_races']).round(0)
pop.drop('n_races', axis=1, inplace=True)

# --- Merge all pieces ---
df = (
    totals
    .merge(race_piv, on=['State', 'State Code', 'Year'])
    .merge(sex_piv,  on=['State', 'State Code', 'Year'])
    .merge(pop,      on=['State', 'State Code', 'Year'])
)

df['rate_per_100k'] = (df['total_deaths'] / df['Population'] * 100_000).round(1)
df['state_abbr']    = df['State Code'].apply(
    lambda x: FIPS_TO_ABBR.get(str(int(x)).zfill(2), '??')
)

# Identify race columns for reuse later
RACE_COLS = [
    c for c in df.columns
    if c.startswith('deaths_') and c not in ('deaths_Male', 'deaths_Female')
]

print(f'Analysis frame: {df.shape}')
print(f'Race columns  : {RACE_COLS}')
print(f'\nSample — Mississippi 2022:')
sample_cols = ['State','Year','total_deaths','rate_per_100k'] + RACE_COLS + ['deaths_Male','deaths_Female']
print(df[(df['State']=='Mississippi') & (df['Year']==2022)][sample_cols].to_string(index=False))

NameError: name 'df_long' is not defined

In [6]:
# ── CELL 5: HOVER TEXT BUILDER ───────────────────────────────────────────────
# Every map calls make_hover() — update this one function to change all tooltips

def make_hover(row, cause_data=None):
    """
    Build the full hover tooltip for one state-year row.
    `cause_data` is an optional dict {'homicide': n, 'suicide': n, 'unintentional': n}
    which will be populated once you add the CDC WONDER cause-type file.
    """
    total = int(row['total_deaths'])
    lines = [
        f"<b>{row['State']}</b>  ·  {int(row['Year'])}",
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━",
        f"Total gun deaths  :  <b>{total:,}</b>",
        f"Rate per 100k     :  <b>{row['rate_per_100k']:.1f}</b>",
        f"Population        :  {int(row['Population']):,}",
    ]

    # ── Death type breakdown (populated when FILE_CAUSE is available) ──────
    if cause_data:
        lines += ['', '<b>── By type of death ──</b>']
        for dtype, n in cause_data.items():
            if n and n > 0:
                pct = n / total * 100
                lines.append(f"  {dtype:<20}:  {int(n):>5,}  ({pct:.1f}%)")
    else:
        lines += [
            '',
            '<i>Death-type breakdown (homicide/suicide/</i>',
            '<i>unintentional) — add CDC WONDER cause file</i>',
            '<i>to FILE_CAUSE in Cell 1 to enable.</i>',
        ]

    # ── Race breakdown ───────────────────────────────────────────────────────
    lines += ['', '<b>── By race ──</b>']
    for col, label in RACE_LABELS.items():
        if col in row.index and pd.notna(row[col]) and row[col] > 0:
            n   = int(row[col])
            pct = n / total * 100
            lines.append(f"  {label:<34}:  {n:>5,}  ({pct:.1f}%)")

    # ── Sex breakdown ────────────────────────────────────────────────────────
    lines += ['', '<b>── By sex ──</b>']
    for col, label in [('deaths_Male', 'Male'), ('deaths_Female', 'Female')]:
        if col in row.index and pd.notna(row[col]) and row[col] > 0:
            n   = int(row[col])
            pct = n / total * 100
            lines.append(f"  {label:<34}:  {n:>5,}  ({pct:.1f}%)")

    lines.append('<i>Source: CDC WONDER Underlying Cause of Death</i>')
    return '<br>'.join(lines)


df['hover'] = df.apply(make_hover, axis=1)

# Quick readability check
test = df[(df['State']=='Louisiana') & (df['Year']==2022)].iloc[0]
print(test['hover']
      .replace('<br>','\n')
      .replace('<b>','').replace('</b>','')
      .replace('<i>','').replace('</i>',''))

NameError: name 'df' is not defined

In [7]:
# ── CELL 6: MAP A — ANIMATED CHOROPLETH (year slider 1999→2024) ──────────────
df_anim = df.dropna(subset=['rate_per_100k','state_abbr']).copy()
df_anim['Year_str'] = df_anim['Year'].astype(str)   # plotly needs string for animation_frame
ZMAX = float(df_anim['rate_per_100k'].quantile(0.97))

fig_anim = px.choropleth(
    df_anim,
    locations='state_abbr',
    locationmode='USA-states',
    color='rate_per_100k',
    animation_frame='Year_str',
    color_continuous_scale=COLORSCALE,
    range_color=(0, ZMAX),
    scope='usa',
    custom_data=['hover'],
    title='US Firearm Deaths per 100k — All States, 1999–2024  (hover for full breakdown)',
)
fig_anim.update_traces(
    hovertemplate='%{customdata[0]}<extra></extra>',
    marker_line_width=0.8, marker_line_color='white',
)
fig_anim.update_layout(
    height=580,
    margin=dict(l=0, r=0, t=55, b=10),
    paper_bgcolor='white',
    geo=dict(bgcolor='white', showlakes=True, lakecolor='white'),
    coloraxis_colorbar=dict(
        title='Deaths<br>per 100k',
        thicknessmode='pixels', thickness=14,
        lenmode='fraction', len=0.65,
        tickfont_size=11,
    ),
    sliders=[{'currentvalue': {'prefix': 'Year: ', 'font': {'size': 14}}, 'pad': {'t': 10}}],
)
fig_anim.show()
print('Use the slider or ▶ Play button to move through years 1999→2024.')

NameError: name 'df' is not defined

In [8]:
# ── CELL 7: MAP B — SINGLE-YEAR INTERACTIVE MAP ──────────────────────────────
# Change YEAR to any value between 1999 and 2024
YEAR = int(df['Year'].max())

df_yr = df[df['Year'] == YEAR].dropna(subset=['rate_per_100k','state_abbr'])

fig_yr = px.choropleth(
    df_yr,
    locations='state_abbr',
    locationmode='USA-states',
    color='rate_per_100k',
    color_continuous_scale=COLORSCALE,
    range_color=(0, float(df_yr['rate_per_100k'].quantile(0.97)) * 1.05),
    scope='usa',
    custom_data=['hover'],
    title=f'US Firearm Deaths per 100k — {YEAR}  (hover for full breakdown)',
)
fig_yr.update_traces(
    hovertemplate='%{customdata[0]}<extra></extra>',
    marker_line_width=0.8, marker_line_color='white',
)
fig_yr.update_layout(
    height=520,
    margin=dict(l=0, r=0, t=50, b=10),
    paper_bgcolor='white',
    geo=dict(bgcolor='white', showlakes=True, lakecolor='white'),
    coloraxis_colorbar=dict(
        title='Deaths<br>per 100k',
        thicknessmode='pixels', thickness=14,
        lenmode='fraction', len=0.65,
    ),
)
fig_yr.show()

NameError: name 'df' is not defined

### Export the interactive heat map (no-code sharing)
This saves the Plotly map as a standalone HTML file you can upload to GitHub or host with GitHub Pages.


In [ ]:
# Export the single-year heat map to a standalone HTML file
import plotly.io as pio

OUT_HTML = 'gun_violence_heatmap.html'
pio.write_html(fig_yr, file=OUT_HTML, include_plotlyjs='cdn', full_html=True)
print(f'Saved: {OUT_HTML}')

# Optional: export the animated map as well
OUT_HTML_ANIM = 'gun_violence_heatmap_animated.html'
pio.write_html(fig_anim, file=OUT_HTML_ANIM, include_plotlyjs='cdn', full_html=True)
print(f'Saved: {OUT_HTML_ANIM}')


In [9]:
# ── CELL 8: MAPS C — RACE-STRATIFIED (one per race group) ────────────────────
YEAR_RACE = int(df['Year'].max())

for col in RACE_COLS:
    label = RACE_LABELS.get(col, col.replace('deaths_',''))

    # Use most recent year that has populated data for this race group
    df_r = df[df[col].notna() & (df[col] > 0)].copy()
    yr   = df_r['Year'].max()
    df_r = df_r[df_r['Year'] == yr].dropna(subset=['state_abbr']).copy()

    if df_r.empty:
        print(f'Skipping {label} — no data.')
        continue

    df_r['_rate']  = (df_r[col] / df_r['Population'] * 100_000).round(1)

    def _hover(r, c=col, l=label, y=yr):
        n    = int(r[c])
        rate = float(r['_rate'])
        pct  = n / r['total_deaths'] * 100 if r['total_deaths'] > 0 else 0
        return (
            f"<b>{r['State']}</b>  ·  {y}<br>"
            "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━<br>"
            f"Race group          :  <b>{l}</b><br>"
            f"Deaths (this group) :  <b>{n:,}</b><br>"
            f"Rate per 100k       :  <b>{rate:.1f}</b><br>"
            f"Share of state total:  {pct:.1f}%<br>"
            f"State total deaths  :  {int(r['total_deaths']):,}<br>"
            "<i>Source: CDC WONDER</i>"
        )

    df_r['_hover'] = df_r.apply(_hover, axis=1)
    _zmax = float(df_r['_rate'].quantile(0.95)) * 1.1

    fig_r = px.choropleth(
        df_r,
        locations='state_abbr',
        locationmode='USA-states',
        color='_rate',
        color_continuous_scale=RACE_COLORS.get(col, 'Blues'),
        range_color=(0, _zmax),
        scope='usa',
        custom_data=['_hover'],
        title=f'Firearm Deaths per 100k — {label}  ({yr})',
    )
    fig_r.update_traces(
        hovertemplate='%{customdata[0]}<extra></extra>',
        marker_line_width=0.8, marker_line_color='white',
    )
    fig_r.update_layout(
        height=490,
        margin=dict(l=0, r=0, t=50, b=10),
        paper_bgcolor='white',
        geo=dict(bgcolor='white', showlakes=True, lakecolor='white'),
        coloraxis_colorbar=dict(
            title='Rate<br>per 100k',
            thicknessmode='pixels', thickness=14,
            lenmode='fraction', len=0.65,
        ),
    )
    fig_r.show()

NameError: name 'df' is not defined

In [10]:
# ── CELL 9: DEATH-TYPE BREAKDOWN HOOK ────────────────────────────────────────
# Once you have a CDC WONDER CSV grouped by ICD-10 cause, drop it in the folder
# and set FILE_CAUSE in Cell 1.  This cell will then:
#   1. Load and parse the cause file
#   2. Merge onto the main df
#   3. Rebuild hover text with homicide/suicide/unintentional counts

if FILE_CAUSE is None:
    print('Death-type data not yet loaded.')
    print('To enable:')
    print('  1. Go to wonder.cdc.gov → Underlying Cause of Death')
    print('  2. Group by: State, Year, ICD-10 Cause List')
    print('  3. Filter causes: W32-W34 (unintentional), X72-X74 (suicide),')
    print('                    X93-X95 (homicide), Y22-Y24 (undetermined)')
    print('  4. Export CSV and set FILE_CAUSE in Cell 1, then re-run')
else:
    causes_raw = _load_wonder(FILE_CAUSE)
    # Standardise — adjust column name if WONDER uses a different label
    cause_col = [c for c in causes_raw.columns if 'cause' in c.lower() or 'icd' in c.lower()][0]
    cause_map = {
        'Accidental discharge of firearms (W32-W34)' : 'Unintentional',
        'Intentional self-harm by firearms (X72-X74)': 'Suicide',
        'Assault by firearms (X93-X95)'              : 'Homicide',
        'Undetermined (Y22-Y24)'                     : 'Undetermined',
    }
    causes_raw['cause_label'] = causes_raw[cause_col].map(cause_map)
    cause_piv = (
        causes_raw.groupby(['State', 'Year', 'cause_label'])['Deaths']
        .sum().unstack('cause_label').reset_index()
    )
    cause_piv.columns = [
        c if c in ['State','Year'] else f'cause_{c}'
        for c in cause_piv.columns
    ]
    df = df.merge(cause_piv, on=['State','Year'], how='left')

    # Rebuild hover with cause breakdown
    def _hover_with_cause(row):
        cause_data = {
            'Homicide'     : row.get('cause_Homicide'),
            'Suicide'      : row.get('cause_Suicide'),
            'Unintentional': row.get('cause_Unintentional'),
            'Undetermined' : row.get('cause_Undetermined'),
        }
        return make_hover(row, cause_data=cause_data)

    df['hover'] = df.apply(_hover_with_cause, axis=1)
    print('Death-type data merged. Re-run Cells 6–8 to see updated tooltips.')

Death-type data not yet loaded.
To enable:
  1. Go to wonder.cdc.gov → Underlying Cause of Death
  2. Group by: State, Year, ICD-10 Cause List
  3. Filter causes: W32-W34 (unintentional), X72-X74 (suicide),
                    X93-X95 (homicide), Y22-Y24 (undetermined)
  4. Export CSV and set FILE_CAUSE in Cell 1, then re-run


In [11]:
# ── CELL 10: MAPS D — MALE VS FEMALE ─────────────────────────────────────────
YEAR_SEX = int(df['Year'].max())
df_sex   = df[df['Year'] == YEAR_SEX].dropna(subset=['state_abbr']).copy()

for sex_col, label, cscale in [
    ('deaths_Male',   'Male',   'Blues'),
    ('deaths_Female', 'Female', 'RdPu'),
]:
    if sex_col not in df_sex.columns:
        continue
    _df = df_sex.copy()
    _df['_rate']  = (_df[sex_col] / _df['Population'] * 100_000).round(1)

    def _hover(r, sc=sex_col, lb=label):
        n   = int(r[sc]) if pd.notna(r.get(sc)) else 0
        pct = n / r['total_deaths'] * 100 if r['total_deaths'] > 0 else 0
        return (
            f"<b>{r['State']}</b>  ·  {YEAR_SEX}<br>"
            "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━<br>"
            f"Sex                 :  <b>{lb}</b><br>"
            f"Deaths ({lb:<7})   :  <b>{n:,}</b><br>"
            f"Rate per 100k       :  <b>{r['_rate']:.1f}</b><br>"
            f"Share of state total:  {pct:.1f}%<br>"
            f"State total deaths  :  {int(r['total_deaths']):,}<br>"
            "<i>Source: CDC WONDER</i>"
        )

    _df['_hover'] = _df.apply(_hover, axis=1)

    fig_s = px.choropleth(
        _df,
        locations='state_abbr',
        locationmode='USA-states',
        color='_rate',
        color_continuous_scale=cscale,
        range_color=(0, float(_df['_rate'].quantile(0.95)) * 1.1),
        scope='usa',
        custom_data=['_hover'],
        title=f'Firearm Deaths per 100k — {label}  ({YEAR_SEX})',
    )
    fig_s.update_traces(
        hovertemplate='%{customdata[0]}<extra></extra>',
        marker_line_width=0.8, marker_line_color='white',
    )
    fig_s.update_layout(
        height=490,
        margin=dict(l=0, r=0, t=50, b=10),
        paper_bgcolor='white',
        geo=dict(bgcolor='white', showlakes=True, lakecolor='white'),
        coloraxis_colorbar=dict(
            title='Rate<br>per 100k',
            thicknessmode='pixels', thickness=14,
            lenmode='fraction', len=0.65,
        ),
    )
    fig_s.show()

NameError: name 'df' is not defined

In [12]:
# ── CELL 11: CHART — TOP STATES TREND LINE 1999→2024 ─────────────────────────
N_TOP  = 10
top_states = (
    df.groupby('State')['rate_per_100k'].mean()
    .nlargest(N_TOP).index.tolist()
)

df_ts = df[df['State'].isin(top_states)].sort_values('Year')

fig_ts = px.line(
    df_ts, x='Year', y='rate_per_100k', color='State',
    markers=True,
    title=f'Firearm Death Rate per 100k — Top {N_TOP} States (1999–2024)',
    labels={'rate_per_100k': 'Deaths per 100k'},
    hover_data={'State': True, 'Year': True,
                'rate_per_100k': ':.1f', 'total_deaths': ':,'},
)
fig_ts.add_vrect(
    x0=2019.8, x1=2021.2,
    fillcolor='rgba(255,200,0,0.18)', line_width=0,
    annotation_text='COVID-19<br>surge', annotation_position='top left',
    annotation_font_size=10,
)
fig_ts.update_layout(
    height=480,
    hovermode='x unified',
    paper_bgcolor='white', plot_bgcolor='#f8f9fa',
    yaxis=dict(gridcolor='#e0e0e0'),
    xaxis=dict(gridcolor='#e0e0e0', dtick=2),
    legend=dict(bgcolor='rgba(255,255,255,0.85)',
                bordercolor='#ddd', borderwidth=1),
)
fig_ts.show()

NameError: name 'df' is not defined

In [13]:
# ── CELL 12: CHART — BLACK VS WHITE DISPARITY OVER TIME ──────────────────────
# Per-race populations come from the raw long file (one row per state/year/race/sex)

pop_race = (
    df_long
    .groupby(['State', 'Year', 'race'])['Population']
    .first().reset_index()   # population is same for M and F in a group
)

black_pop = pop_race[pop_race['race'] == 'Black or African American'].rename(
    columns={'Population': 'pop_Black'})[['State','Year','pop_Black']]
white_pop = pop_race[pop_race['race'] == 'White'].rename(
    columns={'Population': 'pop_White'})[['State','Year','pop_White']]

disp = (
    df[['State','Year',
        'deaths_Black or African American',
        'deaths_White']]
    .merge(black_pop, on=['State','Year'], how='left')
    .merge(white_pop, on=['State','Year'], how='left')
)

disp['rate_Black'] = disp['deaths_Black or African American'] / disp['pop_Black'] * 100_000
disp['rate_White'] = disp['deaths_White'] / disp['pop_White'] * 100_000

nat = disp.groupby('Year')[['rate_Black','rate_White']].mean().reset_index()
nat['ratio'] = (nat['rate_Black'] / nat['rate_White']).round(2)

fig_d = go.Figure()
fig_d.add_trace(go.Scatter(
    x=nat['Year'], y=nat['rate_Black'],
    name='Black / African American',
    mode='lines+markers', line=dict(color='#d62728', width=2.5),
))
fig_d.add_trace(go.Scatter(
    x=nat['Year'], y=nat['rate_White'],
    name='White',
    mode='lines+markers', line=dict(color='#1f77b4', width=2.5),
))
fig_d.add_trace(go.Scatter(
    x=nat['Year'], y=nat['ratio'],
    name='Disparity ratio  (Black ÷ White)',
    mode='lines', yaxis='y2',
    line=dict(color='#8c564b', dash='dash', width=1.8),
))
fig_d.add_vrect(
    x0=2019.8, x1=2021.2,
    fillcolor='rgba(255,200,0,0.18)', line_width=0,
    annotation_text='COVID-19', annotation_position='top left',
    annotation_font_size=10,
)
fig_d.update_layout(
    title='National Average Gun Death Rate: Black vs. White (1999–2024)',
    height=460,
    hovermode='x unified',
    paper_bgcolor='white', plot_bgcolor='#f8f9fa',
    xaxis=dict(title='Year', dtick=2, gridcolor='#e0e0e0'),
    yaxis=dict(title='Deaths per 100k', gridcolor='#e0e0e0'),
    yaxis2=dict(title='Disparity ratio', overlaying='y', side='right',
                showgrid=False, tickformat='.1f'),
    legend=dict(bgcolor='rgba(255,255,255,0.85)',
                bordercolor='#ddd', borderwidth=1),
)
fig_d.show()

NameError: name 'df_long' is not defined

In [14]:
# ── CELL 13: SUMMARY TABLE ────────────────────────────────────────────────────
latest = int(df['Year'].max())
show   = ['State','total_deaths','rate_per_100k'] + RACE_COLS[:4] + ['deaths_Male','deaths_Female']

for title, ascending in [
    (f'TOP 15 states by firearm death rate ({latest})', False),
    (f'BOTTOM 10 states by firearm death rate ({latest})', True),
]:
    n = 15 if 'TOP' in title else 10
    tbl = (
        df[df['Year'] == latest][show]
        .dropna(subset=['rate_per_100k'])
        .sort_values('rate_per_100k', ascending=ascending)
        .head(n).reset_index(drop=True)
    )
    tbl.index += 1
    print(f'\n=== {title} ===')
    print(tbl.to_string())

NameError: name 'df' is not defined

In [15]:
# ── CELL 14: EXPORT ALL TO HTML ───────────────────────────────────────────────
EXPORT = True   # set False to skip

if EXPORT:
    exports = [
        ('map_animated_1999_2024.html',       fig_anim),
        (f'map_{latest}_single_year.html',    fig_yr),
        ('chart_top_states_trend.html',       fig_ts),
        ('chart_racial_disparity.html',       fig_d),
    ]
    for fname, fig in exports:
        fig.write_html(fname, include_plotlyjs='cdn')
        print(f'Saved: {fname}')
    print('\nOpen any .html file in a browser — fully self-contained, no server needed.')
else:
    print('Export skipped.')

NameError: name 'fig_anim' is not defined

## What to add next

| Layer | Source | How to merge |
|---|---|---|
| **Death type** (homicide / suicide / unintentional) | CDC WONDER → group by ICD-10 cause list | Set `FILE_CAUSE` in Cell 1, re-run Cell 9 |
| **Perpetrator data** | FBI NIBRS / Supplemental Homicide Reports (ICPSR) | Merge on state + year |
| **County-level granularity** | Earlier county CSVs already in project | Swap `locationmode` to `geojson-id` with county FIPS |
| **Socioeconomic overlays** | Census ACS (poverty, income, unemployment) | Merge on state FIPS + year |
| **Gun law tier** | RAND State Firearm Law Database | Merge on state + year |